# PDF-to-JSON RAG: retrieval quality vs runtime cost

This executable case study examines two release snapshots from a local-first Document AI project: a controlled comparison of three retrieval/reranking backends and a required quality gate over public PDFs.

The goal is not to claim that a larger model is automatically better. It is to make the engineering trade-off visible: **retrieval quality, query latency, index-build time, model size, and offline operation**.

**Project links:** [GitHub repository](https://github.com/Evanthel/pdf-to-json-rag) · [Interactive Hugging Face demo](https://huggingface.co/spaces/Evanthel/pdf-to-json-rag-demo)

## System under evaluation

```text
PDF → structured document JSON → section-aware chunks
    → local vector index → retrieval/reranking
    → grounded answer + page/chunk citations
```

The full application handles native text, OCR fallback, document routing, extraction diagnostics, persistent ChromaDB indexes, and answer abstention. This notebook focuses narrowly on the evidence needed to choose a default retrieval backend.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:  # Keeps the repository-side smoke test dependency-light.
    def Markdown(value):
        return value

    def display(value):
        print(value)

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#3559E0", "#34A77B", "#F08A4B"]

def locate_dataset() -> Path:
    local = Path.cwd() / "deploy" / "kaggle" / "dataset"
    if (local / "runtime_backend_comparison.json").exists():
        return local

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        matches = list(kaggle_root.glob("**/runtime_backend_comparison.json"))
        if matches:
            return matches[0].parent

    raise FileNotFoundError(
        "Dataset files not found. Attach 'PDF-to-JSON RAG Benchmark Results' "
        "on Kaggle or run this notebook from the project repository root."
    )

DATA_DIR = locate_dataset()
print(f"Reading benchmark data from: {DATA_DIR}")

In [ ]:
runtime = json.loads((DATA_DIR / "runtime_backend_comparison.json").read_text())
public_pdf = json.loads((DATA_DIR / "public_pdf_benchmark.json").read_text())
backends = pd.read_csv(DATA_DIR / "backend_comparison.csv")
public_summary = pd.read_csv(DATA_DIR / "public_pdf_summary.csv")

assert runtime["source"]["all_pass"] is True
assert public_pdf["results"]["quality_gate_passed"] is True
assert len(backends) == 3

display(Markdown(
    f"**Loaded:** {runtime['source']['case_count']} controlled evaluation cases, "
    f"{runtime['source']['comparison_chunk_count']:,} indexed chunks, and "
    f"{public_pdf['corpus']['pdf_count']} public PDFs."
))

## 1. Backend comparison

All three modes were evaluated over the same 77 cases and 2,071 chunks at `k=5`.

- **Baseline:** deterministic hash-384 embeddings plus lightweight reranking; no model weights.
- **Sentence transformer:** `all-MiniLM-L6-v2` embeddings plus lightweight reranking.
- **Cross-encoder:** hash retrieval followed by `ms-marco-MiniLM-L-6-v2` reranking.

Latency is a single-process measurement on Apple Silicon and includes first-use model loading. Treat the absolute milliseconds as host-specific and the relative difference as directional.

In [ ]:
columns = [
    "mode", "case_count", "pass_rate", "precision_at_5", "recall_at_5", "mrr",
    "avg_query_latency_ms", "p95_query_latency_ms", "index_build_seconds",
    "model_weight_mib", "gpu",
]
comparison = backends[columns].copy()
comparison["pass_rate"] = comparison["pass_rate"].map(lambda value: f"{value:.1%}")
display(comparison.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

quality = backends.set_index("mode")[["recall_at_5", "mrr"]]
quality.plot(kind="bar", ax=axes[0], color=COLORS[:2], width=0.72)
axes[0].set_title("Retrieval quality")
axes[0].set_ylabel("score")
axes[0].set_xlabel("")
axes[0].set_ylim(0.90, 1.01)
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(["Recall@5", "MRR"], frameon=False, loc="lower right")

latency = backends.set_index("mode")[["avg_query_latency_ms", "p95_query_latency_ms"]]
latency.plot(kind="bar", ax=axes[1], color=[COLORS[0], COLORS[2]], width=0.72)
axes[1].set_title("Query latency")
axes[1].set_ylabel("milliseconds")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(["Average", "p95"], frameon=False)

fig.suptitle("Same test set, different runtime cost", fontsize=15, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
baseline_latency = backends.loc[
    backends["mode"] == "baseline", "avg_query_latency_ms"
].iloc[0]
tradeoffs = backends[[
    "mode", "avg_query_latency_ms", "index_build_seconds", "model_weight_mib", "fit"
]].copy()
tradeoffs["query_latency_vs_baseline"] = (
    tradeoffs["avg_query_latency_ms"] / baseline_latency
).round(2)
display(tradeoffs.round(2))

same_quality = (
    backends[["recall_at_5", "mrr", "keyword_coverage"]].nunique() == 1
).all()
cross_multiplier = tradeoffs.loc[
    tradeoffs["mode"] == "cross-encoder", "query_latency_vs_baseline"
].iloc[0]

display(Markdown(
    f"**Observed decision:** quality was {'identical' if same_quality else 'different'} "
    f"on this test set, while the cross-encoder averaged **{cross_multiplier:.2f}×** "
    "the baseline query latency. The snapshot therefore keeps `auto` as the default, "
    "prefers sentence-transformer retrieval when its model is cached, and retains the "
    "hash backend as the dependency-light offline fallback."
))

## 2. Required public-PDF quality gate

The second snapshot comes from a required GitHub Actions shard. It processes three reviewed public PDFs, builds a fresh index, and evaluates six grounded retrieval cases. The original PDFs are deliberately **not redistributed** in this Kaggle dataset; the source repository records their URLs, licenses, and SHA-256 checksums.

No LLM was used in this run, so the result isolates deterministic extraction, retrieval, and evidence-grounding behavior.

In [ ]:
summary_columns = [
    "pdf_count", "case_count", "chunk_count", "pass_rate", "recall_at_5", "mrr",
    "answer_keyword_coverage", "evidence_keyword_coverage", "embedding_backend",
    "llm_used_case_count",
]
display(public_summary[summary_columns].round(3))

observed = {
    "Pass rate": float(public_summary.loc[0, "pass_rate"]),
    "MRR": float(public_summary.loc[0, "mrr"]),
    "Evidence coverage": float(public_summary.loc[0, "evidence_keyword_coverage"]),
}
required = {
    "Pass rate": public_pdf["quality_gate"]["min_pass_rate"],
    "MRR": public_pdf["quality_gate"]["min_mrr"],
    "Evidence coverage": public_pdf["quality_gate"]["min_evidence_keyword_coverage"],
}

gate = pd.DataFrame({"Observed": observed, "Required": required})
ax = gate.plot(kind="bar", figsize=(9, 4.5), color=[COLORS[1], "#AAB2C8"], width=0.72)
ax.set_title("Public-PDF benchmark: observed result vs release gate")
ax.set_ylabel("score")
ax.set_xlabel("")
ax.set_ylim(0, 1.08)
ax.tick_params(axis="x", rotation=0)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## What the results do — and do not — show

**Supported by these snapshots**

- The deterministic fallback is a credible default for this portfolio-scale corpus.
- Adding a sentence-transformer improved semantics available to the system without changing measured quality on this particular set, but it added model download and index-build cost.
- Cross-encoder reranking added the highest latency and did not improve these aggregate metrics, so it remains opt-in.
- The release gate is reproducible in CI and fails when processing or retrieval quality drops below explicit thresholds.

**Limitations**

- 77 controlled cases and six public-PDF cases are useful regression evidence, not a universal ranking benchmark.
- Exact latency depends on hardware, process warm-up, model cache state, and corpus size.
- Aggregate retrieval metrics can hide per-document errors; the main project also stores case-level diagnostics.
- The snapshot does not evaluate generated prose from a hosted LLM.

## Reproduce the full benchmark

The compact files attached to this notebook are immutable release snapshots. To rerun extraction and retrieval from source:

```bash
git clone https://github.com/Evanthel/pdf-to-json-rag.git
cd pdf-to-json-rag
python -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[semantic]'
pdf-to-json-rag compare-runtime-modes   --modes baseline,sentence-transformers,cross-encoder   --all-cases --json
```

The repository documents the public corpus, integrity checks, CI gates, and security limits. For the product flow itself, open the [Hugging Face Space](https://huggingface.co/spaces/Evanthel/pdf-to-json-rag-demo) and use a public, non-confidential PDF.

## Conclusion

The result is intentionally pragmatic: **a more complex retrieval backend must earn its runtime and operational cost on measured quality**. For this release, it did not. The project therefore keeps a deterministic offline path, makes semantic retrieval available when cached, and treats the cross-encoder as an explicit experiment rather than an automatic upgrade.